# Chokri Bidirectional Fine-tuning
**First:** Runtime → Change runtime type → T4 GPU → Save

This notebook auto-resumes after disconnects — just run all cells again.

In [ ]:
import torch
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory/1e9,1), 'GB')
    print('GPU ready')
else:
    print('No GPU detected.')
    print('Go to: Runtime > Change runtime type > T4 GPU > Save')
    raise SystemExit('Please enable GPU before continuing.')


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# Edit these paths if needed
DRIVE_ROOT  = '/content/drive/MyDrive/Chokri'
DRIVE_DATA  = DRIVE_ROOT + '/data'
DRIVE_MODEL = DRIVE_ROOT + '/model_best'
DRIVE_CKPT  = DRIVE_ROOT + '/checkpoints'
TOTAL_EPOCHS = 8
BATCH_SIZE   = 4
GRAD_ACCUM   = 8
LR           = 3e-5
WARMUP_STEPS = 300
print('Config set.')
print('Drive model path: ' + DRIVE_MODEL)
print('Drive data path:  ' + DRIVE_DATA)


In [ ]:
import subprocess
subprocess.run(['pip', 'install', '-q', 'transformers', 'sentencepiece', 'sacrebleu'])
print('Dependencies ready.')


In [ ]:
import os, shutil, subprocess
os.makedirs('/content/data', exist_ok=True)
for fname in ['train.csv', 'val.csv', 'test.csv']:
    src = DRIVE_DATA + '/' + fname
    dst = '/content/data/' + fname
    if os.path.exists(src):
        shutil.copy(src, dst)
        result = subprocess.run(['wc', '-l', dst], capture_output=True, text=True)
        n = int(result.stdout.split()[0]) - 1
        print(fname + ': ' + str(n) + ' rows')
    else:
        print('WARNING: not found: ' + src)


In [ ]:
import os, csv, math, time, json
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, get_linear_schedule_with_warmup

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

CHOKRI_TOKEN  = "zho_Hant"
ENGLISH_TOKEN = "eng_Latn"
MAX_LEN       = 128
PROGRESS_FILE = DRIVE_ROOT + "/training_progress.json"
OPT_FILE      = DRIVE_ROOT + "/optimizer_state.pt"
BASE_CKPT     = "facebook/nllb-200-distilled-600M"
device        = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

def load_progress():
    if os.path.exists(PROGRESS_FILE):
        with open(PROGRESS_FILE) as f:
            return json.load(f)
    return {"completed_epochs": 0, "best_val_loss": 9999.0}

def save_progress(completed, best_loss):
    with open(PROGRESS_FILE, "w") as f:
        json.dump({"completed_epochs": completed, "best_val_loss": best_loss}, f)

progress         = load_progress()
completed_epochs = progress["completed_epochs"]
best_val_loss    = progress["best_val_loss"]

if completed_epochs >= TOTAL_EPOCHS:
    print("Training already complete: " + str(completed_epochs) + "/" + str(TOTAL_EPOCHS) + " epochs")
    print("To restart: upload a fresh training_progress.json with completed_epochs=0")
else:
    if completed_epochs == 0:
        print("Starting fresh from base NLLB model")
        start_from = BASE_CKPT
    else:
        print("Resuming from epoch " + str(completed_epochs + 1) + "/" + str(TOTAL_EPOCHS))
        print("Best val loss so far: " + str(round(best_val_loss, 4)))
        start_from = DRIVE_MODEL

    def load_pairs(path):
        fwd, rev = [], []
        with open(path, newline="", encoding="utf-8") as f:
            for row in csv.DictReader(f):
                c = row["chokri"].strip()
                e = row["english"].strip()
                if c and e:
                    fwd.append((CHOKRI_TOKEN, c, ENGLISH_TOKEN, e))
                    rev.append((ENGLISH_TOKEN, e, CHOKRI_TOKEN, c))
        combined = []
        for a, b in zip(fwd, rev):
            combined.append(a)
            combined.append(b)
        return combined

    class BilingualDataset(Dataset):
        def __init__(self, ex): self.ex = ex
        def __len__(self): return len(self.ex)
        def __getitem__(self, i): return self.ex[i]

    def collate_fn(batch, tok):
        src_langs, srcs, tgt_langs, tgts = zip(*batch)
        pad_id = tok.pad_token_id
        inp_ids, att_masks, labels = [], [], []
        for sl, s, tl, t in zip(src_langs, srcs, tgt_langs, tgts):
            tok.src_lang = sl
            enc = tok(s, max_length=MAX_LEN, truncation=True, return_tensors="pt")
            inp_ids.append(enc["input_ids"][0])
            att_masks.append(enc["attention_mask"][0])
            tok.src_lang = tl
            lab = tok(t, max_length=MAX_LEN, truncation=True, return_tensors="pt")
            lids = lab["input_ids"][0].clone()
            lids[lids == pad_id] = -100
            labels.append(lids)
        def pad(seqs, val):
            ml = max(s.size(0) for s in seqs)
            return torch.stack([torch.cat([s, torch.full((ml - s.size(0),), val, dtype=s.dtype)]) for s in seqs])
        return {
            "input_ids":      pad(inp_ids, pad_id),
            "attention_mask": pad(att_masks, 0),
            "labels":         pad(labels, -100),
        }

    print("Loading model from: " + start_from)
    tok   = AutoTokenizer.from_pretrained(start_from)
    model = AutoModelForSeq2SeqLM.from_pretrained(start_from)
    model.gradient_checkpointing_enable()
    model.to(device)
    print("Model ready.")

    train_ex = load_pairs("/content/data/train.csv")
    val_ex   = load_pairs("/content/data/val.csv")
    print("Train: " + str(len(train_ex)) + " examples  Val: " + str(len(val_ex)) + " examples")

    fn           = lambda b: collate_fn(b, tok)
    train_loader = DataLoader(BilingualDataset(train_ex), batch_size=BATCH_SIZE, shuffle=True,  collate_fn=fn, num_workers=2, pin_memory=True)
    val_loader   = DataLoader(BilingualDataset(val_ex),   batch_size=BATCH_SIZE, shuffle=False, collate_fn=fn, num_workers=2, pin_memory=True)

    remaining   = TOTAL_EPOCHS - completed_epochs
    total_steps = math.ceil(len(train_loader) / GRAD_ACCUM) * TOTAL_EPOCHS
    opt         = torch.optim.AdamW(model.parameters(), lr=LR)
    sch         = get_linear_schedule_with_warmup(opt, WARMUP_STEPS, total_steps)
    scaler      = torch.amp.GradScaler("cuda")

    # Restore optimizer state if resuming
    if completed_epochs > 0 and os.path.exists(OPT_FILE):
        print("Restoring optimizer state...")
        opt_state = torch.load(OPT_FILE, map_location=device)
        opt.load_state_dict(opt_state["optimizer"])
        sch.load_state_dict(opt_state["scheduler"])
        scaler.load_state_dict(opt_state["scaler"])
        print("Optimizer state restored.")
    elif completed_epochs > 0:
        print("Warning: no optimizer state found, resuming without it.")

    os.makedirs(DRIVE_CKPT, exist_ok=True)
    global_step = completed_epochs * math.ceil(len(train_loader) / GRAD_ACCUM)

    for epoch in range(completed_epochs + 1, TOTAL_EPOCHS + 1):
        model.train()
        epoch_loss, n = 0.0, 0
        opt.zero_grad()
        t0 = time.time()

        for step, batch in enumerate(train_loader, 1):
            batch = {k: v.to(device) for k, v in batch.items()}
            with torch.amp.autocast("cuda"):
                loss = model(**batch).loss / GRAD_ACCUM
            scaler.scale(loss).backward()
            epoch_loss += loss.item() * GRAD_ACCUM
            n += 1
            if step % GRAD_ACCUM == 0:
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(opt)
                scaler.update()
                sch.step()
                opt.zero_grad()
                torch.cuda.empty_cache()
                global_step += 1

        # Validate
        model.eval()
        vl, vn = 0.0, 0
        with torch.no_grad():
            for batch in val_loader:
                batch = {k: v.to(device) for k, v in batch.items()}
                with torch.amp.autocast("cuda"):
                    vl += model(**batch).loss.item()
                vn += 1
        val_loss = vl / max(vn, 1)
        elapsed  = time.time() - t0
        print("Epoch " + str(epoch) + "/" + str(TOTAL_EPOCHS) +
              "  train=" + str(round(epoch_loss/n, 4)) +
              "  val="   + str(round(val_loss, 4)) +
              "  time="  + str(round(elapsed)) + "s")

        # Save model checkpoint to Drive
        ckpt_path = DRIVE_CKPT + "/epoch_" + str(epoch)
        model.save_pretrained(ckpt_path)
        tok.save_pretrained(ckpt_path)
        print("  checkpoint saved -> " + ckpt_path)

        # Save best model
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            model.save_pretrained(DRIVE_MODEL)
            tok.save_pretrained(DRIVE_MODEL)
            print("  best model saved (val_loss=" + str(round(best_val_loss, 4)) + ")")

        # Save optimizer state for clean resume
        torch.save({
            "optimizer": opt.state_dict(),
            "scheduler": sch.state_dict(),
            "scaler":    scaler.state_dict(),
        }, OPT_FILE)

        # Save progress
        save_progress(epoch, best_val_loss)
        print("  progress saved (" + str(epoch) + "/" + str(TOTAL_EPOCHS) + " complete)")

    print("Training complete! Best val loss: " + str(round(best_val_loss, 4)))
    print("Best model at: " + DRIVE_MODEL)


In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tok   = AutoTokenizer.from_pretrained(DRIVE_MODEL)
model = AutoModelForSeq2SeqLM.from_pretrained(DRIVE_MODEL).to(device)
model.eval()

eng_id    = tok.convert_tokens_to_ids("eng_Latn")
chokri_id = tok.convert_tokens_to_ids("zho_Hant")

print("=== Chokri -> English ===")
for s in ["Abraham nu David, David nuo Jisu Khrista shühye lüsida;",
          "i lawva ti-va", "chekha kha-te"]:
    tok.src_lang = "zho_Hant"
    inp = tok(s, return_tensors="pt", max_length=128, truncation=True).to(device)
    with torch.no_grad():
        out = model.generate(**inp, forced_bos_token_id=eng_id, num_beams=4, max_new_tokens=128, no_repeat_ngram_size=3, repetition_penalty=1.5)
    print("  Chokri : " + s)
    print("  English: " + tok.decode(out[0], skip_special_tokens=True))
    print()

print("=== English -> Chokri ===")
for s in ["Close the door.", "I will eat food.", "Good people go to heaven."]:
    tok.src_lang = "eng_Latn"
    inp = tok(s, return_tensors="pt", max_length=128, truncation=True).to(device)
    with torch.no_grad():
        out = model.generate(**inp, forced_bos_token_id=chokri_id, num_beams=4, max_new_tokens=128, no_repeat_ngram_size=3, repetition_penalty=1.5)
    print("  English: " + s)
    print("  Chokri : " + tok.decode(out[0], skip_special_tokens=True))
    print()


In [ ]:
import sacrebleu, csv, torch

test_pairs = []
with open("/content/data/test.csv", newline="", encoding="utf-8") as f:
    for row in csv.DictReader(f):
        test_pairs.append((row["chokri"].strip(), row["english"].strip()))

srcs   = [p[0] for p in test_pairs]
refs   = [p[1] for p in test_pairs]
hyps   = []
eng_id = tok.convert_tokens_to_ids("eng_Latn")

for i in range(0, len(srcs), 16):
    batch = srcs[i:i+16]
    tok.src_lang = "zho_Hant"
    inp = tok(batch, return_tensors="pt", max_length=128, truncation=True, padding=True).to(device)
    with torch.no_grad():
        out = model.generate(**inp, forced_bos_token_id=eng_id, num_beams=4, max_new_tokens=128, no_repeat_ngram_size=3, repetition_penalty=1.5)
    hyps.extend(tok.batch_decode(out, skip_special_tokens=True))
    if (i//16+1) % 5 == 0:
        print(str(i+len(batch)) + "/" + str(len(srcs)) + " done...")

bleu = sacrebleu.corpus_bleu(hyps, [refs])
print("BLEU score (Chokri->English): " + str(round(bleu.score, 2)))
